# 06 — Plan B : recherche Sentinel-1 sur Douala (tâche D6)

Objectif : trouver des images **Sentinel-1** (radar bande C) pour avoir une **référence en saison sèche**
(déc.–févr.), absente de NISAR sur les pistes qui couvrent bien Douala.
Rien n'est téléchargé : on dresse un inventaire `inventaire_s1_douala.csv`.

⚠️ Sentinel-1 est en **bande C** (NISAR : bande L). Les valeurs en dB ne sont pas équivalentes ;
le seuil d'eau du contrat (HH < -18 dB) devra être recalibré pour VV/VH (contrat v2.0, accord de l'équipe).

In [ ]:
!pip -q install asf_search pandas shapely

## 1. Paramètres

In [ ]:
import json, os
import pandas as pd
import asf_search as asf
from shapely.geometry import shape, box

# Bbox du contrat de données v1.0 (celle utilisée par le tableau de bord)
BBOX = box(9.55, 3.90, 9.90, 4.20)

# Grande zone D1 (cercle approché), pour information
AOI = box(9.2721, 3.69052, 9.9386, 4.35538)
if os.path.exists("douala_aoi.geojson"):
    with open("douala_aoi.geojson", encoding="utf-8") as fh:
        AOI = shape(json.load(fh)["features"][0]["geometry"])
    print("Grande zone lue depuis douala_aoi.geojson")

START = "2025-11-01"
END   = "2026-11-15"

# On interroge sur la bbox du contrat : c'est la zone qui compte pour le tableau de bord.
print("Surface bbox (deg2) :", round(BBOX.area, 4), "| grande zone :", round(AOI.area, 4))

## 2. Recherche (aucun téléchargement)

On demande des produits **GRD haute résolution** en mode **IW** : légers (~1 Go) et suffisants pour cartographier l'eau.

In [ ]:
results = asf.geo_search(
    platform=[asf.PLATFORM.SENTINEL1],
    intersectsWith=BBOX.wkt,
    start=START,
    end=END,
    processingLevel=[asf.PRODUCT_TYPE.GRD_HD],
    beamMode=asf.BEAMMODE.IW,
    maxResults=2000,
)
print(f"{len(results)} scènes Sentinel-1 GRD_HD / IW trouvées sur la bbox")

Si le résultat est **0**, retirez `processingLevel` et `beamMode`, puis relancez, ou vérifiez sur https://search.asf.alaska.edu (Dataset = Sentinel-1).

In [ ]:
if results:
    props = results[0].properties
    for k in sorted(props):
        print(f"{k:28} {str(props[k])[:80]}")

## 3. Inventaire

In [ ]:
def first(d, *keys):
    for k in keys:
        v = d.get(k)
        if v not in (None, ""):
            return v
    return None

def couverture(geom, zone):
    if geom is None or not geom.is_valid:
        return None
    return round(100 * geom.intersection(zone).area / zone.area, 1)

rows = []
for r in results:
    p = r.properties
    geom = shape(r.geometry) if r.geometry else None
    rows.append({
        "date":         (first(p, "startTime") or "")[:10],
        "start_time":   first(p, "startTime"),
        "scene":        first(p, "sceneName", "fileID"),
        "satellite":    first(p, "platform"),
        "produit":      first(p, "processingLevel"),
        "mode":         first(p, "beamModeType"),
        "orbite_dir":   first(p, "flightDirection"),
        "piste":        str(first(p, "pathNumber", "relativeOrbit")),
        "polarisation": first(p, "polarization"),
        "taille_MB":    round(first(p, "bytes") / 1e6) if isinstance(first(p, "bytes"), (int, float)) else None,
        "couv_bbox_%":  couverture(geom, BBOX),
        "couv_zone_%":  couverture(geom, AOI),
        "url":          first(p, "url"),
    })

df = pd.DataFrame(rows)
if not df.empty:
    df = df.sort_values("date").reset_index(drop=True)
df.head(15)

## 4. Saison et résumés

In [ ]:
def saison(d):
    return "seche" if int(d[5:7]) in (12, 1, 2) else "pluies"

if df.empty:
    print("Tableau vide : rien à résumer.")
else:
    df["saison"] = df["date"].map(saison)
    print("Période :", df["date"].min(), "->", df["date"].max())
    for titre, cols in [("Par satellite", ["satellite"]), ("Par polarisation", ["polarisation"]),
                        ("Par piste / sens", ["piste", "orbite_dir"])]:
        print(); print(titre); print(df.groupby(cols, dropna=False).size().to_string())
    print(); print("Par mois :"); print(df.groupby(df["date"].str[:7]).size().to_string())

## 5. Groupes comparables (même piste + même sens + bbox couverte)

In [ ]:
COUV_MIN = 99   # % de la bbox couverte

if df.empty:
    groupes = pd.DataFrame()
else:
    ok = df[df["couv_bbox_%"] >= COUV_MIN]
    groupes = (ok.groupby(["piste", "orbite_dir", "polarisation"], dropna=False)
                 .apply(lambda g: pd.Series({
                     "n_dates":  g["date"].nunique(),
                     "n_seche":  g.loc[g.saison == "seche",  "date"].nunique(),
                     "n_pluies": g.loc[g.saison == "pluies", "date"].nunique()}))
                 .reset_index())
    groupes["utilisable"] = (groupes.n_seche > 0) & (groupes.n_pluies > 0)
    groupes = groupes.sort_values(["utilisable", "n_dates"], ascending=False).reset_index(drop=True)
groupes

## 6. Dates proposées (meilleur groupe : 2 sèches + 3 des pluies)

In [ ]:
if groupes.empty or not groupes.utilisable.any():
    print("Aucun groupe avec saison sèche ET saison des pluies : élargir START/END ou baisser COUV_MIN.")
else:
    best = groupes[groupes.utilisable].iloc[0]
    g = ok[(ok.piste == best.piste) & (ok.orbite_dir == best.orbite_dir) & (ok.polarisation == best.polarisation)]
    g = g.drop_duplicates("date")
    seche  = g[g.saison == "seche"].sort_values("date")
    pluies = g[g.saison == "pluies"].sort_values("date")
    # sèches : au coeur de la saison ; pluies : début, milieu, fin de la série (étalées)
    import numpy as np
    pick = lambda d, n: d.iloc[np.unique(np.linspace(0, len(d) - 1, min(n, len(d))).round().astype(int))]
    choix = pd.concat([pick(seche, 2), pick(pluies, 3)]).sort_values("date")
    print("Groupe retenu :", dict(best[["piste", "orbite_dir", "polarisation"]]))
    print(choix[["date", "saison", "satellite", "piste", "orbite_dir", "polarisation", "couv_bbox_%", "taille_MB", "scene"]].to_string())

## 7. Sauvegarde

In [ ]:
df.to_csv("inventaire_s1_douala.csv", index=False)
if "choix" in globals():
    choix.to_csv("dates_retenues_s1.csv", index=False)
print("Enregistrés : inventaire_s1_douala.csv" + (" et dates_retenues_s1.csv" if "choix" in globals() else ""))

## Fini quand (D6)
- [ ] Au moins 1 groupe avec des dates **sèches ET des pluies**, même piste et même sens
- [ ] `inventaire_s1_douala.csv` et `dates_retenues_s1.csv` partagés avec l'équipe
- [ ] L'équipe a décidé du seuil d'eau pour VV/VH (contrat v2.0)